# Social Behavior Processing Pipeline

**Project:** Social Single/Group Housing Behavior Analysis  
**Author:** Mir Qi 
**Last Updated:** October 20, 2024

## Overview
This notebook processes and validates multi-camera behavioral recordings for social interaction experiments. The pipeline includes:
1. Scanning and logging folder structures
2. Reading and consolidating parquet logs
3. Data quality checks and filtering
4. Visualization of COM (center of mass) trajectories
5. DANNCE 3D pose validation

---
## 1. Setup & Configuration

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
sys.path.append(os.path.abspath('../..'))

# Import custom modules
from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

In [2]:
# Configuration
BASE_FOLDER = "/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused"
FAILED_PATHS_FILE = os.path.join(BASE_FOLDER, "bad_calib.txt")

# Rescan settings
FORCE_RESCAN_SESSIONS = []  # Add specific sessions to force rescan: [('2023-10-01', '001'), ...]
RESCAN_THRESHOLD_DAYS = 0.000000001  # Days threshold for automatic rescan

---
## 2. Scan Folders & Generate Logs

This step scans the data directory structure and creates/updates parquet log files for each session.

In [3]:
# Scan all folders and create logs
log_folder_to_parquet_sep(
    BASE_FOLDER,
    FAILED_PATHS_FILE,
    STATUS_FIELDS_CONFIG,
    force_rescan_rec_files=FORCE_RESCAN_SESSIONS,
    rescan_threshold_days=RESCAN_THRESHOLD_DAYS
)

print("\n✓ Folder scanning complete")

Log for 0single5_group3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_17/0single5_group3/folder_log.parquet
Log for 0single4_group4 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_17/0single4_group4/folder_log.parquet
Log for 0single1_group1 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_09_26/0single1_group1/folder_log.parquet
Log for 0single2_group2 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_09_26/0single2_group2/folder_log.parquet
Log for 0single4_group4 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_09_26/0single4_group4/folder_log.parquet
Log for 0single3_group3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_09_26/0single3_group3/folder_log.parquet
Log for extrinsics saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03_17_00/extrinsics/folder_log.parquet
Log for 0single1_group1 saved a

---
## 3. Load & Consolidate Data

Read all parquet files - keep PyArrow table for filtering, create pandas for display.

In [4]:
# Load all parquet logs - KEEP AS PYARROW TABLE
all_table = read_all_parquet_files(BASE_FOLDER)

# Create pandas version ONLY for display
all_df = all_table.to_pandas()

print(f"Total sessions loaded: {len(all_df)}")
print(f"Columns: {list(all_df.columns)}")

Total sessions loaded: 35
Columns: ['mir_generate_param', 'sync', 'mini_6cam_map', 'dropf_handle', 'com', 'com_vis', 'social', 'miniscope', 'test', 'after_oxytocin', 'before_oxytocin', 'dannce', 'dannce_vis', 'mini_rec_sync', 'mini_rec_sync_com', 'rec_file', 'scan_time', 'rec_path', 'date_folder', 'calib_files']


---
## 4. Data Overview & Quality Checks

In [5]:
print(f"Dataset Shape: {all_df.shape}\n")
print("First few rows:")
all_df.head()

Dataset Shape: (35, 20)

First few rows:


,mir_generate_param,sync,mini_6cam_map,dropf_handle,com,com_vis,social,miniscope,test,after_oxytocin,before_oxytocin,dannce,dannce_vis,mini_rec_sync,mini_rec_sync_com,rec_file,scan_time,rec_path,date_folder,calib_files
0,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single5_group2,2025-10-31T16:05:32.764432,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
1,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single3_group4,2025-10-31T16:05:32.772685,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
2,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0single4_group3,2025-10-31T16:05:32.805624,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
3,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single2_group5,2025-10-31T16:05:32.847568,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
4,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single5_group2_1,2025-10-31T16:05:32.848034,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."


In [6]:
print("\nProcessing Status Summary:")
status_cols = ['sync', 'com', 'com_vis', 'dannce', 'dannce_vis']

for col in status_cols:
    if col in all_df.columns:
        completed = (all_df[col] == 1).sum()
        print(f"  {col}: {completed}/{len(all_df)} completed")


Processing Status Summary:
  sync: 0/35 completed
  com: 0/35 completed
  com_vis: 0/35 completed
  dannce: 0/35 completed
  dannce_vis: 0/35 completed


---
## 5. Filter Data

Apply filters to select sessions for processing. **Use PyArrow table for filtering!**

In [7]:
import pyarrow.compute as pc

# FILTER ON PYARROW TABLE (not pandas)
mask = (
    pc.equal(all_table['social'], 1) &
    pc.equal(all_table['com'], 1) &
    pc.equal(all_table['com_vis'], 0)
)

# Apply filter to get PyArrow table
filtered_table = all_table.filter(mask)

# NOW convert to pandas for display
filtered_df = filtered_table.to_pandas()

print(f"\nFiltered sessions: {len(filtered_df)}")
print(f"\nSessions to process:")
filtered_df[['date_folder', 'rec_file', 'com', 'com_vis']]

ArrowNotImplementedError: Function 'equal' has no kernel matching input types (string, int64)

---
## 6. Visualize Session Timeline

Show timeline of sessions with preview images.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from datetime import datetime

def visualize_sessions_timeline(table_or_df, base_folder, max_sessions=15):
    """
    Visualize sessions in chronological order with preview images.
    
    Args:
        table_or_df: PyArrow Table or Pandas DataFrame
        base_folder: Base path to data
        max_sessions: Maximum sessions to display
    """
    # Convert to pandas if needed for visualization
    if hasattr(table_or_df, 'to_pandas'):
        df = table_or_df.to_pandas()
    else:
        df = table_or_df
    
    if len(df) == 0:
        print("No sessions to visualize")
        return
    
    # Limit number of sessions
    df_display = df.head(max_sessions).copy()
    
    # Sort by scan_time
    df_display = df_display.sort_values('scan_time')
    
    # Create figure
    n_sessions = len(df_display)
    cols = 5
    rows = (n_sessions + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
    axes = axes.flatten() if n_sessions > 1 else [axes]
    
    for idx, (_, row) in enumerate(df_display.iterrows()):
        ax = axes[idx]
        
        rec_name = row['rec_file']
        date_folder = row['date_folder']
        mtime = row['scan_time']
        
        # Try to find preview image
        session_path = os.path.join(base_folder, date_folder, rec_name)
        img_path = os.path.join(session_path, "imgs", "cam1_frame_000000.jpg")
        
        if os.path.isfile(img_path):
            img = mpimg.imread(img_path)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "No Image", ha="center", va="center", 
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
        
        timestamp = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d\n%H:%M")
        ax.set_title(f"{rec_name}\n{timestamp}", fontsize=8)
        ax.axis("off")
    
    # Hide unused subplots
    for idx in range(n_sessions, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.suptitle("Session Timeline (Chronological)", fontsize=14, y=1.02)
    plt.show()

# Visualize filtered sessions - can use either filtered_table or filtered_df
visualize_sessions_timeline(filtered_df, BASE_FOLDER, max_sessions=15)

---
## 7. Generate COM Visualizations

Process filtered sessions to generate center-of-mass trajectory visualizations for social experiments.

In [ ]:
from utlis.vis_valid_utlis.scom_traga_utlis import plot_com_all_social

# Convert filtered PyArrow table to list of records
# Use .to_pylist() for PyArrow table OR iterate over pandas DataFrame
if hasattr(filtered_table, 'to_pylist'):
    # PyArrow method
    records = [
        {
            'date_folder': row['date_folder'].as_py(),
            'rec_file': row['rec_file'].as_py()
        }
        for row in filtered_table
    ]
else:
    # Pandas method
    records = [
        {
            'date_folder': row['date_folder'],
            'rec_file': row['rec_file']
        }
        for _, row in filtered_df.iterrows()
    ]

print(f"Processing {len(records)} sessions...\n")

# Process each session
for i, record in enumerate(records, 1):
    session_path = os.path.join(BASE_FOLDER, record['date_folder'], record['rec_file'])
    print(f"[{i}/{len(records)}] {session_path}")
    
    try:
        plot_com_all_social(
            session_path,
            perform_generate_com_video=True
        )
        print("  ✓ Complete")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

print("\n=== COM visualization complete ===")

---
## 8. Validate DANNCE 3D Poses

Run validation on DANNCE 3D pose estimation results.

In [ ]:
from useful_files.sophie_check_dannce_mir_modif import dannce_valid
from concurrent.futures import ProcessPoolExecutor, as_completed

# Filter for sessions needing DANNCE validation - USE PYARROW TABLE
dannce_mask = (
    pc.equal(all_table['dannce'], 1) &
    pc.equal(all_table['dannce_vis'], 0)
)
dannce_table = all_table.filter(dannce_mask)

# Convert to records
records = [
    {
        'date_folder': row['date_folder'].as_py(),
        'rec_file': row['rec_file'].as_py()
    }
    for row in dannce_table
]

print(f"Processing {len(records)} DANNCE sessions...\n")

def process_dannce_session(record):
    """Process a single DANNCE session with error handling."""
    session_path = os.path.join(BASE_FOLDER, record['date_folder'], record['rec_file'])
    try:
        dannce_valid(session_path)
        return f"✓ {session_path}"
    except Exception as e:
        return f"✗ {session_path}: {e}"

# Parallel processing
with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_dannce_session, record) for record in records]
    
    for future in as_completed(futures):
        result = future.result()
        print(result)

print("\n=== DANNCE validation complete ===")

---
## 9. Summary & Next Steps

In [ ]:
# Final summary
print("\n" + "="*50)
print("PROCESSING SUMMARY")
print("="*50)
print(f"Total sessions: {len(all_df)}")
print(f"Sessions processed: {len(filtered_df)}")
print(f"\nNext steps:")
print("  1. Review generated visualizations")
print("  2. Check for any failed sessions")
print("  3. Update processing flags in database")
print("="*50)

---
## Notes & Documentation

### Key Files
- **STATUS_FIELDS_CONFIG**: Defines processing pipeline stages
- **bad_calib.txt**: List of sessions with calibration issues

### Processing Stages
1. `mir_generate_param` - Mirror parameter generation
2. `sync` - Camera synchronization
3. `mini_6cam_map` - 6-camera mapping
4. `dropf_handle` - Dropped frame handling
5. `com` - Center of mass calculation
6. `com_vis` - COM visualization
7. `social` - Social interaction flags
8. `dannce` - 3D pose estimation
9. `dannce_vis` - DANNCE visualization

### Troubleshooting
- If sessions are missing, check file permissions
- For failed visualizations, verify image paths
- Parallel processing may require adjusting worker count based on system resources

### Data Type Strategy
- Keep `all_table` as PyArrow for **filtering** (fast, efficient)
- Convert to pandas (`all_df`, `filtered_df`) only for **display/visualization**
- Use `.as_py()` when iterating over PyArrow tables
- Use regular pandas iteration when working with DataFrames